# 혼잡도(CROWD) EDA 탐색 노트북 — 141

88~135에서 확정된 분석 결과를 **셀 단위로 바꿔 보며** 확인하는 노트북. 그림 코드는 여기 없고
`DATA_ENGINE/eda/report_figures.py`의 `fig_*`를 호출만 한다 — 호선·역·요일유형·타깃을 인자로 바꾸면
그 자리에서 다시 그려진다. 공유용 그림(PNG)은 `python -m DATA_ENGINE.eda.report_figures`가 만들고,
수치의 원본은 `validation/CROWD/**/RESULTS.md`다. 두 곳이 어긋나면 RESULTS.md가 맞다.

**규칙** — 커밋 전 출력을 비운다(`AI/CLAUDE.md`). CI ruff가 ipynb도 검사한다. 탐색에서 결론이 나면
함수로 옮겨 `report_figures.py`에 추가하고 여기는 호출만 남긴다.

| 절 | 무엇을 보나 | 출처 |
| --- | --- | --- |
| 1 | 입력 패널 — 호선별 총량 추이, 역 × 시간대 히트맵 | 88 |
| 2 | 혼잡도 라벨 — 방향 분해 검증, 배율표 | 88 |
| 3 | 시간 해상도 — 30분 비중, 열차별 배분, 배차 분포 | 135 |
| 4 | 잔차 — 쏠림, 역별 시계열, **탐색: 서울역·6호선·요일유형** | 87·90 |
| 5 | 모델 — 세트별 개선율, 등급 임계치 | 89·90 |

In [ ]:
# 실행 위치: AI/ (커널 cwd가 AI/가 아니면 아래에서 맞춘다)
import os
from pathlib import Path

if Path.cwd().name != "AI":
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "app").is_dir() and (cand / "DATA_ENGINE").is_dir():
            os.chdir(cand)
            break
import sys

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import pandas as pd

from DATA_ENGINE.eda import figstyle as fs
from DATA_ENGINE.eda import report_figures as rf

font = fs.apply(inline=True)  # 파일 저장 대신 셀에 그린다
print("cwd:", Path.cwd(), "| 폰트:", font or "없음(라벨 깨짐 주의)")
inp = rf.Inputs()  # 무거운 입력은 한 번만 읽어 아래 셀이 공유

### 선택에 쓰는 이름들
- 호선: `rf.LINE_ORDER` = 1~8호선. 9호선은 패널 기간이 달라 제외.
- 요일유형: `rf.DAY_TYPES` = 평일·토요일·일요일·휴일.
- 역: `station_no`로 고른다. 서울역처럼 호선마다 번호가 다른 역이 있다(1호선 150, 4호선 426).

In [ ]:
stations = inp.stations()
print(rf.LINE_ORDER, rf.DAY_TYPES)
stations[stations["station_name"].str.contains("서울역|강남|종합운동장|잠실", regex=True)]

## 1. 입력 패널 (88)

**결론** — 2호선이 하루 140만 명으로 압도적이고 설·추석·연말에 전 호선이 같은 폭으로 꺼진다.
평일 08시·18시 두 봉우리는 전 호선 공통. 요일유형 lookup이 베이스라인으로 강한 이유가 이 규칙성이다.

In [ ]:
rf.fig_daily_total_by_line(inp);  # lines=["2호선", "5호선"], start="2025-01-01", target="alighting"

In [ ]:
# 호선 하나만 고르면 역 이름이 축에 붙는다. day_types를 늘리면 장수가 늘어난다.
rf.fig_panel_heatmap(inp, lines=["2호선"], day_types=["평일"]);

## 2. 혼잡도 라벨 (88)

**결론** — 재귀식 raw 혼잡도와 실측 30분 스냅샷의 양의 상관(3호선 0.88, 2·4호선 0.80)이 방향 라벨이
맞다는 근거. 1호선은 절단 구간이라 0.38. 배율표는 배차 보정과 30분 안 모양을 함께 담고, 종점·첫차
구간(raw≈0)에서 배율이 튄다.

In [ ]:
rf.fig_direction_validation(inp);  # lines=["2호선", "6호선"], day_type="토요일"

In [ ]:
rf.fig_calibration_ratio_heatmap(inp, lines=["6호선"]);  # directions=("상선",) / day_type="일요일"

## 3. 시간 해상도 (135)

**결론** — 1시간 안 후반 30분 비중은 7시 57%, 8시 45%, 19시 44%로 역·요일 불문 안정(러시 σ 2.6~3.7%p).
열차 배분은 직전 간격 비례이며 러시 배차 3~5분이라 무작위 도착 가정이 유효. 5호선 지선(12분 초과 10.8%)과
심야는 `headway_long` 플래그를 봐야 한다. 열차별 값은 **추정**이다.

In [ ]:
rf.fig_half_hour_share_curve(inp);
rf.fig_half_hour_share_curve(inp, stations=[222], day_types=["평일", "일요일"]);  # 한 역만: 띠는 방향 간 산포

In [ ]:
rf.fig_train_load(inp);  # station_no=150, direction="하선", date="2025-06-04", slots=("18:00", "18:30")

In [ ]:
rf.fig_headway_distribution(inp, lines=["2호선", "5호선", "8호선"]);  # day_type="일요일", long_headway_min=10

## 4. 잔차 (87·90)

잔차 = 2025 실측 − 2024 lookup(요일유형×역×시간대 평균). 모델이 줄여야 하는 것이 이 잔차다.

**결론** — 서울역(1호선) 한 곳이 전체 제곱오차의 약 14%, 종합운동장 13%. 시간대는 21~22·17~19·08시,
요일유형은 토요일이 행 비중보다 오차 비중이 크다.

In [ ]:
rf.fig_residual_concentration(inp);  # lines=["6호선"], top_n=10, target="alighting"

In [ ]:
rf.fig_station_residual_timeseries(inp, stations=[150, 426, 218]);  # 서울역 1호선·4호선, 종합운동장

### 탐색 A — 서울역 잔차는 1호선만의 현상인가

90에서 "이벤트로 설명되지 않는 서울역 양의 추세"가 미해결로 남았다. 같은 역의 4호선 쪽(426)과 비교하면
원인이 역(유동)인지 노선(집계)인지 갈린다.

In [ ]:
r = rf.residuals(inp.panel)  # split="2025-01-01", target="boarding"
r["month"] = r["date"].dt.to_period("M")
seoul = (
    r[r["station_no"].isin([150, 426])]
    .groupby(["station_no", "month"])["resid"]
    .mean()
    .unstack(0)
    .rename(columns={150: "서울역(1호선)", 426: "서울역(4호선)"})
)
ax = seoul.plot(kind="bar", figsize=(11, 4), color=[fs.LINE_COLORS["1호선"], fs.LINE_COLORS["4호선"]])
ax.axhline(0, color=fs.PALETTE_NEUTRAL[1], lw=0.8)
ax.set_ylabel("월평균 잔차 (명/시간대)")
ax.set_title("서울역 — 1호선은 연중 +400~+1,200명/시간대로 커지고, 4호선은 0 근처")
seoul.round(0)

**읽기** — 4호선 서울역은 잔차가 0 근처인데 1호선만 연중 커진다. 역 주변 유동(이벤트·공사)이면 두 호선이
함께 움직여야 하므로, **1호선 서울역 집계 자체의 변화**(2025년 집계 범위·환승 처리 변경 등)일 가능성이
크다. → 93에서 1호선 서울역은 2025 원천 정의를 확인한 뒤 다루고, 그 전까지는 lookup 학습 창을 2025로
좁히는 것(롤링)이 이 역의 처방이다.

### 탐색 B — 6호선은 왜 모델 개선율이 가장 낮은가(+11%)

lookup 잔차의 크기(RMSE)를 그 호선 승차의 산포(표준편차)로 나눈 값이 크면, "요일유형·시간대 평균으로
설명되지 않는 몫"이 크다는 뜻이다.

In [ ]:
rmse = r.groupby("line", observed=True)["resid"].agg(lambda x: float((x**2).mean() ** 0.5))
std = inp.panel[inp.panel["date"] >= "2025-01-01"].groupby("line", observed=True)["boarding"].std()
tbl = pd.DataFrame({"lookup 잔차 RMSE": rmse, "승차 표준편차": std})
tbl["잔차/산포"] = (tbl["lookup 잔차 RMSE"] / tbl["승차 표준편차"]).round(2)
tbl = tbl.reindex(rf.LINE_ORDER).round(1)
ax = tbl["잔차/산포"].plot(kind="bar", figsize=(9, 3.5), color=[fs.line_color(x) for x in tbl.index])
ax.set_ylabel("lookup 잔차 RMSE ÷ 승차 표준편차")
ax.set_title("호선별 '평균으로 설명되지 않는 몫' — 1·6호선이 높다")
tbl

**읽기** — 6호선은 절대 잔차는 작지만(승차 자체가 적다) 산포 대비 잔차 비율이 1호선과 함께 가장 높다.
규칙성이 약해 lookup도, 자기 시차 모델도 덜 먹는다. 93(호선·방향 분리 학습)에서 6호선을 첫 후보로 둔
근거가 이 표다. 6호선 안에서 어느 역인지는 `rf.fig_residual_concentration(inp, lines=["6호선"])`.

### 탐색 C — 요일유형별 잔차는 어느 시간대에 몰리나

90의 "주말·공휴일은 평일의 60% 수준" 뒤에 있는 모양. 전날 시차가 다른 요일유형을 가리키는 문제와 연결.

In [ ]:
mat = (
    r.assign(abs_resid=r["resid"].abs())
    .groupby(["day_type", "time_slot"], observed=True)["abs_resid"]
    .mean()
    .unstack("time_slot")
    .reindex(index=rf.DAY_TYPES, columns=rf.SLOT_ORDER)
)
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(mat.to_numpy(), aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(rf.SLOT_ORDER)))
ax.set_xticklabels(rf.SLOT_ORDER, rotation=90)
ax.set_yticks(range(len(rf.DAY_TYPES)))
ax.set_yticklabels(rf.DAY_TYPES)
ax.grid(False)
fig.colorbar(im, ax=ax, label="평균 |잔차| (명)")
ax.set_title("요일유형 × 시간대 평균 절대 잔차 — 평일은 러시, 주말·휴일은 오후~저녁에 몰린다")
mat.round(0)

**읽기** — 평일 잔차는 08시·18시 러시에, 주말·휴일 잔차는 오후~저녁에 넓게 퍼진다. 주말은 러시 규칙이
없어 "같은 요일유형의 직전 날" 시차(B′)가 평일보다 더 필요하다는 방향. → 93 후보 피처.

## 5. 모델 (89·90)

입력이 없으면 빈 리스트가 돌아온다 — `compare_models.py --save-results`, `grade_sensitivity.py --save-cells`로
만든다(각 5~10분).

**결론** — D−1에서 쓸 수 있는 자기 시차 세트가 +21~24%, 실시간 집계가 필요한 인접역 세트와 같은 크기.
국토부 임계치(150/170/190)는 99.6%가 한 등급이라 판별력이 없고, 50/100에서 lookup 94.0 → 모델 95.1%.

In [ ]:
rf.fig_feature_set_steps(inp) or print("compare_results.parquet 없음");  # model="lightgbm"

In [ ]:
rf.fig_grade_threshold_sensitivity(inp) or print("grade_cells.parquet 없음");
# thresholds={"50/100": [50, 100], "40/90": [40, 90]}, lines=["2호선"]

---
### 새 탐색을 시작할 때
1. 위 `r`(잔차)·`inp.panel`·`inp.calibration`을 바로 쓴다. 다시 읽지 않는다.
2. 결론이 나면 `report_figures.py`에 `fig_*`로 옮기고 이 노트북에는 호출 한 줄만 남긴다.
3. 수치는 `RESULTS.md`, 요약은 Notion 실험실 페이지에. 커밋 전 `출력 지우기` + `ruff check DATA_ENGINE/eda/crowd_eda.ipynb`.